# 05 — Active Aero ΔCdA: Data Availability

This notebook documents the investigation into whether active aero drag reduction
(ΔCdA) can be measured from Canada 2026 telemetry, and why it currently cannot.

**Background**: In 2026 F1 replaced DRS with an automatic active aero system.
Cars deploy straight mode (low drag) on designated straights without any gap-to-car
requirement, eliminating the slipstream confound that made the 2024 DRS measurement
ambiguous.  In principle this should give a clean ΔCdA measurement from coast-down
segments.  In practice the data is not available.

In [ ]:
import sys, os, sqlite3, pickle, json, base64, zlib
sys.path.insert(0, '..')
import fastf1
import numpy as np
os.makedirs('../cache', exist_ok=True)
fastf1.Cache.enable_cache('../cache')

## Channel scan: is active aero state in the F1 timing API?

In [ ]:
def scan_drs(sess, label):
    """Report DRS/active-aero channel values across all drivers in a session."""
    all_vals = set()
    drivers_nonzero = []
    for drv, df in sess.car_data.items():
        if 'DRS' not in df.columns:
            continue
        vals = sorted(df['DRS'].unique().tolist())
        all_vals.update(vals)
        n = int((df['DRS'] > 10).sum())
        if n > 0:
            drivers_nonzero.append((drv, vals, n, len(df)))
    print(f'{label}')
    print(f'  Drivers: {len(sess.car_data)}  |  All DRS values: {sorted(all_vals)}')
    print(f'  Drivers with DRS > 10: {len(drivers_nonzero)}')
    if not drivers_nonzero:
        print('  → channel all-zero for every driver')
    print()

# 2026 Race (main session for this analysis)
sess_2026r = fastf1.get_session(2026, 'Canada', 'Race')
sess_2026r.load(telemetry=True, weather=False)
scan_drs(sess_2026r, 'Canada 2026 Race')

# 2026 FP1 (only practice session available on Sprint weekend)
sess_2026fp = fastf1.get_session(2026, 'Canada', 'FP1')
sess_2026fp.load(telemetry=True, weather=False)
scan_drs(sess_2026fp, 'Canada 2026 FP1')

# Sanity check: 2024 Monza where DRS was definitely used
sess_2024 = fastf1.get_session(2024, 'Monza', 'Race')
sess_2024.load(telemetry=True, weather=False)
scan_drs(sess_2024, 'Monza 2024 Race  ← sanity check, should be non-zero')

# Raw API channel inspection: what channel keys does 2026 car data actually contain?
print('=== Raw API channel keys ===')
CACHE_DB = '../cache/fastf1_http_cache.sqlite'
conn = sqlite3.connect(CACHE_DB)
cur  = conn.cursor()
cur.execute('SELECT value FROM responses ORDER BY length(value) DESC LIMIT 20')

for (blob,) in cur.fetchall():
    obj = pickle.loads(blob)
    if 'CarData' not in obj.get('url', ''):
        continue
    year = '2026' if '2026' in obj['url'] else '2024'
    content = obj['_content'].decode('utf-8-sig')
    ch_keys = set()
    for line in content.split('\r\n')[:3000]:
        parts = line.strip().split('"', 1)
        if len(parts) < 2:
            continue
        try:
            raw   = zlib.decompress(base64.b64decode(parts[1]), -zlib.MAX_WBITS)
            entry = json.loads(raw.decode('utf-8-sig'))
            if 'Entries' in entry:
                for e in entry['Entries']:
                    for drv in e.get('Cars', {}).values():
                        ch_keys.update(drv.get('Channels', {}).keys())
        except Exception:
            continue
    if ch_keys:
        label = f"{year} {'Race' if 'Race' in obj['url'] else 'FP1'}"
        print(f'  {label}: channels = {sorted(ch_keys, key=lambda x: int(x) if x.isdigit() else 99)}')

conn.close()
print()
print('Channel 45 = DRS in 2024.  Absent from 2026 means the signal is not broadcast.')

## Conclusion

The F1 live timing API does not broadcast active aero state for 2026 sessions.
Channel 45 (DRS in 2024) is absent from every 2026 car data message.
The five channels present in 2026 — RPM (0), Speed (2), nGear (3), Throttle (4),
Brake (5) — are identical to 2024 minus channel 45.

This is not a FastF1 bug. The FastF1 source (`_api.py` line 1022) already notes:
```python
drs = entry['Cars'][drv]['Channels'].get('45', 0)
# drs is no longer included in 2026
```

**Why this matters**: Without knowing which segments have active aero deployed, any
comparison of coast-down drag between "straight" and "corner" segments conflates the
aero mode difference with speed, gear, and track-position differences.  The result
would not be a measurement of ΔCdA.

**What would make this measurement feasible**:
1. F1 adds active aero state to their live timing API (analogous to channel 45)
2. FastF1 decodes it — the architecture already supports it; only the channel
   number and value encoding need updating
3. The approach in `src/segments.py` (`segment_active_aero_state`) will then
   automatically use the telemetry signal instead of the position fallback